# Real-Time AI Agent Interfaces with AG-UI on Amazon Bedrock AgentCore

| Information | Details |
|:------------|:--------|
| Agent Framework | Strands Agents |
| LLM | Anthropic Claude Sonnet 4 |
| Components | AgentCore Runtime, AG-UI event streaming, S3/CloudFront frontend |
| CLI | AgentCore CLI (`@aws/agentcore`) |

This notebook deploys a Strands agent to AgentCore that emits real AG-UI events — tool calls, reasoning, text streaming — viewable in a browser frontend.

**Prerequisites**: SageMaker Studio, AWS credentials with Bedrock access, Node.js 20+, supporting infrastructure (Lambda proxy + S3/CloudFront) already deployed.

In [13]:
%pip install -U typing_extensions strands-agents strands-agents-tools strands-agents-bedrock boto3 uv --quiet
!command -v agentcore >/dev/null || npm install -g @aws/agentcore --silent
!agentcore --version
!command -v zip >/dev/null || (sudo apt-get install -y zip --quiet 2>/dev/null || brew install zip 2>/dev/null || echo 'install zip manually')
#!which uv && which zip


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
0.14.0


In [14]:
# Sanity-check toolchain versions. If anything is missing or unexpected, install/upgrade before continuing.
import sys, subprocess, importlib.metadata as md

def cli_version(cmd):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        return (p.stdout or p.stderr).strip().splitlines()[0]
    except FileNotFoundError:
        return 'NOT INSTALLED'
    except Exception as e:
        return f'error: {e}'

def pkg_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return 'NOT INSTALLED'

print('=== CLIs ===')
print(f'  agentcore     : {cli_version(["agentcore", "--version"])}')
print(f'  node          : {cli_version(["node", "--version"])}')
print(f'  npm           : {cli_version(["npm", "--version"])}')
print(f'  aws cli       : {cli_version(["aws", "--version"])}')
print(f'  uv            : {cli_version(["uv", "--version"])}')
print(f'  zip           : {cli_version(["zip", "-v"]).split()[1] if cli_version(["zip", "-v"]) != "NOT INSTALLED" else "NOT INSTALLED"}')

print('\n=== Python packages ===')
print(f'  python                 : {sys.version.split()[0]}')
for pkg in ['boto3', 'botocore', 'strands-agents', 'strands-agents-tools', 'strands-agents-bedrock', 'fastapi', 'uvicorn']:
    print(f'  {pkg:<22} : {pkg_version(pkg)}')

=== CLIs ===
  agentcore     : 0.14.0
  node          : v24.2.0
  npm           : 11.6.3
  aws cli       : aws-cli/2.27.40 Python/3.13.5 Darwin/25.4.0 source/arm64
  uv            : uv 0.11.14 (3fdfdc7d4 2026-05-12 aarch64-apple-darwin)
  zip           : (c)

=== Python packages ===
  python                 : 3.14.4
  boto3                  : 1.43.9
  botocore               : 1.43.9
  strands-agents         : 1.40.0
  strands-agents-tools   : 0.5.3
  strands-agents-bedrock : 0.0.1
  fastapi                : 0.136.1
  uvicorn                : 0.47.0


## Configuration

All tunable values live here. Change `MODEL_ID` to switch foundation models, `AGENT_NAME` to deploy alongside an existing agent, etc. Every later cell reads from these variables.

In [15]:
import os
from boto3.session import Session

# ── Project identity ────────────────────────────────────────────────
AGENT_NAME = 'aguisampleagent'             # [a-zA-Z][a-zA-Z0-9_]{0,47}
PROJECT_NAME = 'aguisampleagent'           # max 23 chars, alphanumeric, starts with letter
STACK_NAME = 'agui-bedrock-agentcore'      # Supporting infra (Lambda + S3 + CloudFront)
LAMBDA_FUNCTION_NAME = 'agui-bedrock-agentcore-proxy'

# ── Foundation model ────────────────────────────────────────────────
# Bedrock inference profile id. "global." prefix routes to whichever region has capacity.
# Other options: 'us.anthropic.claude-3-5-haiku-20241022-v1:0', 'global.anthropic.claude-opus-4-7', etc.
MODEL_ID = 'global.anthropic.claude-sonnet-4-6'

# ── Agent behavior ──────────────────────────────────────────────────
SYSTEM_PROMPT = (
    'You are a business operations assistant. Use multiple tools when needed. '
    'Always look up data before recommending. Calculate numbers, dont estimate. '
    'If you find issues, send alerts. Be concise.'
)

# ── Paths (anchored to repo root; robust to os.chdir) ──────────────
# Walk upward from cwd to find a directory containing the notebook itself.
# This makes paths immune to `os.chdir(PROJECT_DIR)` later in the notebook.
NOTEBOOK_FILE = 'agui_bedrock_agentcore_deploy.ipynb'
_search = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.isfile(os.path.join(_search, NOTEBOOK_FILE)):
        break
    parent = os.path.dirname(_search)
    if parent == _search:
        raise RuntimeError(f'Could not locate {NOTEBOOK_FILE!r} from {os.getcwd()!r}')
    _search = parent
NOTEBOOK_DIR = _search
PROJECT_DIR = os.path.join(NOTEBOOK_DIR, AGENT_NAME)
AGENT_FILE = os.path.join(NOTEBOOK_DIR, 'agui_agent.py')
CFN_TEMPLATE = os.path.join(NOTEBOOK_DIR, 'deploy', 'cloudformation.yaml')
FRONTEND_HTML = os.path.join(NOTEBOOK_DIR, 'frontend', 'index.html')

# ── AWS context ─────────────────────────────────────────────────────
_session = Session()
AWS_REGION = _session.region_name or 'us-east-1'
AWS_ACCOUNT_ID = _session.client('sts').get_caller_identity()['Account']

print(f'AGENT_NAME    : {AGENT_NAME}')
print(f'STACK_NAME    : {STACK_NAME}')
print(f'MODEL_ID      : {MODEL_ID}')
print(f'AWS_REGION    : {AWS_REGION}')
print(f'AWS_ACCOUNT   : {AWS_ACCOUNT_ID}')
print(f'NOTEBOOK_DIR  : {NOTEBOOK_DIR}')

AGENT_NAME    : aguisampleagent
STACK_NAME    : agui-bedrock-agentcore
MODEL_ID      : global.anthropic.claude-sonnet-4-6
AWS_REGION    : us-east-1
AWS_ACCOUNT   : <ACCOUNT-ID>
NOTEBOOK_DIR  : /Users/skoppar/workspace/sample-agui-bedrock-agentcore


## Step 1: The AG-UI agent server

The agent code lives in [`agui_agent.py`](agui_agent.py) at the repo root — it's the single source of truth. It's a FastAPI server that wraps a Strands agent with 7 tools and uses `agent.stream_async()` to translate Strands events into real-time AG-UI events as the model produces them. AgentCore exposes it on `/invocations` (SSE) and `/ping` (health).

The deploy cell below copies this file into the AgentCore project's `app/` directory at deploy time, so any edits you make to `agui_agent.py` flow through on the next deploy.

In [16]:
# Source of truth: the agent code lives in agui_agent.py at the repo root.
# This cell just confirms the file is present and surfaces its size/hash so you
# know what version will be deployed by the `launch` cell below.
import hashlib

with open(AGENT_FILE, 'rb') as f:
    body = f.read()

print(f'Path : {AGENT_FILE}')
print(f'Size : {len(body):,} bytes')
print(f'SHA1 : {hashlib.sha1(body).hexdigest()[:12]}')

Path : /Users/skoppar/workspace/sample-agui-bedrock-agentcore/agui_agent.py
Size : 14,929 bytes
SHA1 : 7869ad92fd57


## Step 2: Test the agent locally

In [18]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def test_calc(expression: str) -> dict:
    """Calculate math.
    Args:
        expression: Math expression
    """
    return {"result": eval(expression, {"__builtins__": {}})}

model = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)
test_agent = Agent(model=model, tools=[test_calc], system_prompt='Always use the calculator tool for math.')
result = test_agent('What is 42 * 17?')
print(result)


Tool #1: test_calc
42 * 17 = **714**42 * 17 = **714**



## Step 3: Deploy to AgentCore Runtime

`agentcore create` scaffolds the project and generates `agentcore/agentcore.json`, then `agentcore deploy` packages and deploys to AgentCore.

In [20]:
import shutil

# IMPORTANT: chdir back to NOTEBOOK_DIR before rmtree, so we don't yank the
# rug out from under the running Python process. If a prior `launch` cell
# did `os.chdir(PROJECT_DIR)`, our cwd points at the directory we're about
# to delete — and Node's process.cwd() blows up with ENOENT when we shell
# out to `agentcore`.
os.chdir(NOTEBOOK_DIR)

# Clean up previous project folder if it exists
if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
    print(f'Removed existing {AGENT_NAME}/')

!agentcore create --name {AGENT_NAME} --project-name {PROJECT_NAME} --framework Strands --model-provider Bedrock --memory none


[done]  Create aguisampleagent/ project directory
[done]  Prepare agentcore/ directory
[done]  Initialize git repository
[done]  Add agent to project
[done]  Set up Python environment

Created:
  aguisampleagent/
    app/aguisampleagent/  Python agent (Strands)
    agentcore/            Config and CDK project

Project created successfully!

To continue, navigate to your new project:

  cd aguisampleagent
  agentcore



In [21]:
import shutil, json, re

# cd into project dir
os.chdir(PROJECT_DIR)

# New CLI layout: code lives under app/<agent-name>/, config is agentcore/agentcore.json
code_dir = os.path.join('app', AGENT_NAME)

# Copy our agent code in as main.py (replaces the scaffolded stub)
shutil.copy2(AGENT_FILE, os.path.join(code_dir, 'main.py'))

# Add fastapi + uvicorn to the scaffolded pyproject.toml (the FastAPI server in agui_agent.py needs them)
pyproject_path = os.path.join(code_dir, 'pyproject.toml')
with open(pyproject_path) as f:
    pp = f.read()
for dep in ['"fastapi"', '"uvicorn[standard]"']:
    if dep not in pp:
        pp = re.sub(r'(dependencies = \[)', r'\1\n    ' + dep + ',', pp, count=1)
with open(pyproject_path, 'w') as f:
    f.write(pp)

# Register a 'default' deployment target so `agentcore deploy` knows which account/region to deploy to
targets_path = 'agentcore/aws-targets.json'
with open(targets_path) as f:
    targets = json.load(f)
if not any(t.get('name') == 'default' for t in targets):
    targets.append({'name': 'default', 'account': AWS_ACCOUNT_ID, 'region': AWS_REGION})
    with open(targets_path, 'w') as f:
        json.dump(targets, f, indent=2)

# Update agentcore.json: set region on the runtime entry, inject env vars so the deployed
# agent reads MODEL_ID and SYSTEM_PROMPT from its container environment.
config_path = 'agentcore/agentcore.json'
with open(config_path) as f:
    config = json.load(f)
rt = config['runtimes'][0]
rt.setdefault('aws', {})['region'] = AWS_REGION

desired_env = {'MODEL_ID': MODEL_ID, 'SYSTEM_PROMPT': SYSTEM_PROMPT}
existing = {e['name']: e for e in rt.get('envVars', [])}
for name, value in desired_env.items():
    existing[name] = {'name': name, 'value': value}
rt['envVars'] = list(existing.values())

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Account:       {AWS_ACCOUNT_ID}')
print(f'Region:        {AWS_REGION}')
print(f'Model:         {MODEL_ID}')
print(f"Entrypoint:    {rt['entrypoint']}")
print(f"Code location: {rt['codeLocation']}")
print(f"Runtime:       {rt['runtimeVersion']}")
!agentcore deploy --target default -y

Account:       <ACCOUNT-ID>
Region:        us-east-1
Model:         global.anthropic.claude-sonnet-4-6
Entrypoint:    main.py
Code location: app/aguisampleagent/
Runtime:       PYTHON_3_14
✓ Load deployment target
✓ Validate project...
✓ Build CDK project...
✓ Synthesize CloudFormation...
✓ Check bootstrap status...
✓ Check stack status...
✓ Deploy to AWS...
ersist deployment state......✓ Persist deployment state

✓ Deployed to 'default' (stack: AgentCore-aguisampleagent-default)

Outputs:
  ApplicationAgentAguisampleagentRuntimeIdOutput01A18508: aguisampleagent_aguisampleagent-esfpXICGVN
  ApplicationAgentAguisampleagentRoleArnOutputA86EF783: arn:aws:iam::<ACCOUNT-ID>:role/AgentCore-aguisampleagent-ApplicationAgentAguisampl-HRJlW6gNfZAf
  ApplicationAgentAguisampleagentRuntimeArnOutput722E9B9B: arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT-ID>:runtime/aguisampleagent_aguisampleagent-esfpXICGVN
  StackNameOutput: AgentCore-aguisampleagent-default

Note: Transaction search enabled. It ta

In [22]:
!agentcore status --target default

AgentCore Status (target: default, us-east-1)

Agents
  aguisampleagent: Deployed - Runtime: READY (arn:aws:bedrock-agentcore:us-east-
1:<ACCOUNT-ID>:runtime/aguisampleagent_aguisampleagent-esfpXICGVN)
  URL: https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abe
drock-agentcore%3Aus-east-1%3A<ACCOUNT-ID>%3Aruntime%2Faguisampleagent_aguisampl
eagent-esfpXICGVN/invocations


## Step 4: Test on AgentCore

In [23]:
!agentcore invoke --target default "Check warehouse-east for low stock items and send a slack alert"

nvoking agent...
Session: 1f23e5df-18c5-4c8a-bc1a-23d3814d5479
To resume: agentcore invoke --session-id 1f23e5df-18c5-4c8a-bc1a-23d3814d5479
Log: /Users/skoppar/workspace/sample-agui-bedrock-agentcore/aguisampleagent/agentcore/.cli/logs/invoke/invoke-aguisampleagent-20260515-214044.log


In [24]:
import boto3, json, subprocess, re

INVOKE_PROMPT = 'Look up customer CUST-001 and CUST-003, compare their annual spend, generate a report'

def get_agent_arn(target='default'):
    """Run `agentcore status --json` and extract the agent ARN robustly."""
    proc = subprocess.run(
        ['agentcore', 'status', '--target', target, '--type', 'agent', '--json'],
        capture_output=True, text=True
    )
    if proc.returncode != 0:
        raise RuntimeError(f'agentcore status failed: {proc.stderr or proc.stdout}')
    # The CLI may append non-JSON trailing output (TTY artifacts, log paths, etc.).
    # Extract the first complete top-level JSON object from stdout.
    m = re.search(r'\{.*\}', proc.stdout, re.DOTALL)
    if not m:
        raise RuntimeError(f'No JSON found in status output: {proc.stdout!r}')
    status = json.loads(m.group(0))

    agent_entry = next((r for r in status.get('resources', []) if r.get('resourceType') == 'agent'), None)
    if not agent_entry:
        raise RuntimeError(f'No agent resource in status output: {status}')
    if agent_entry.get('deploymentState') != 'deployed':
        raise RuntimeError(f"Agent is in state '{agent_entry.get('deploymentState')}', not 'deployed'. Run the deploy cell first.")
    # The ARN is stored under the 'identifier' field in this CLI version.
    arn = agent_entry.get('identifier') or agent_entry.get('arn') or agent_entry.get('agentRuntimeArn')
    if not arn:
        raise RuntimeError(f'No ARN found in agent entry: {agent_entry}')
    return arn, agent_entry

agent_arn, agent_entry = get_agent_arn()
print(f'Agent ARN: {agent_arn}')
print(f'State:     {agent_entry.get("deploymentState")} ({agent_entry.get("detail")})')

# Invoke via boto3 for raw SSE inspection
client = boto3.client('bedrock-agentcore', region_name=AWS_REGION)

resp = client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier='DEFAULT',
    payload=json.dumps({'prompt': INVOKE_PROMPT})
)

print(f'\nContent-Type: {resp.get("contentType", "unknown")}\n')
ct = resp.get('contentType', '')
if 'text/event-stream' in ct:
    for line in resp['response'].iter_lines(chunk_size=1):
        if line:
            d = line.decode('utf-8')
            if d.startswith('data: '):
                ev = json.loads(d[6:])
                extra = {k:v for k,v in ev.items() if k not in ('type','eventId','timestamp')}
                print(f"[{ev['type']}] {json.dumps(extra) if extra else ''}")
else:
    for chunk in resp.get('response', []):
        print(chunk.decode('utf-8') if isinstance(chunk, bytes) else chunk)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT-ID>:runtime/aguisampleagent_aguisampleagent-esfpXICGVN
State:     deployed (READY)

Content-Type: text/event-stream; charset=utf-8

[RUN_STARTED] {"runId": "ab2f4913-00ff-4c4a-977b-1ad8080c0683"}
[STATE_SNAPSHOT] {"snapshot": {"status": "processing", "tools_invoked": []}}
[TEXT_MESSAGE_START] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "role": "assistant"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "delta": "Sure"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "delta": "!"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "delta": " I'll start"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "delta": " by looking up both"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d0f3a96b0e", "delta": " customers simultaneously"}
[TEXT_MESSAGE_CONTENT] {"messageId": "c542ea28-7dbf-4500-93e6-78d

## Step 5: Wire up the frontend

Deploy the supporting CloudFormation stack (Lambda proxy + S3 + CloudFront), then update the proxy with the agent ARN and print the frontend URL.

In [32]:
import subprocess, hashlib, os, shutil

PROXY_DIR = os.path.join(NOTEBOOK_DIR, 'deploy', 'proxy')
PROXY_ZIP = os.path.join(NOTEBOOK_DIR, 'deploy', '.build', 'proxy.zip')
ASSETS_BUCKET = f'{STACK_NAME}-assets-{AWS_REGION}-{AWS_ACCOUNT_ID}'

def stack_status():
    p = subprocess.run(
        ['aws', 'cloudformation', 'describe-stacks', '--stack-name', STACK_NAME,
         '--region', AWS_REGION, '--query', 'Stacks[0].StackStatus', '--output', 'text'],
        capture_output=True, text=True
    )
    return p.stdout.strip() if p.returncode == 0 else None

def ensure_assets_bucket():
    """Create the bucket that holds the proxy Lambda zip if it doesn't exist."""
    head = subprocess.run(['aws', 's3api', 'head-bucket', '--bucket', ASSETS_BUCKET], capture_output=True, text=True)
    if head.returncode == 0:
        return
    print(f'Creating assets bucket s3://{ASSETS_BUCKET} ...')
    args = ['aws', 's3api', 'create-bucket', '--bucket', ASSETS_BUCKET]
    if AWS_REGION != 'us-east-1':
        args += ['--create-bucket-configuration', f'LocationConstraint={AWS_REGION}']
    args += ['--region', AWS_REGION]
    subprocess.run(args, check=True)
    subprocess.run(['aws', 's3api', 'put-public-access-block', '--bucket', ASSETS_BUCKET,
                    '--public-access-block-configuration',
                    'BlockPublicAcls=true,IgnorePublicAcls=true,BlockPublicPolicy=true,RestrictPublicBuckets=true'], check=True)

def build_proxy_zip():
    """npm install in deploy/proxy and zip the result. Returns (s3_key, content_hash)."""
    if not os.path.exists(os.path.join(PROXY_DIR, 'node_modules')):
        print('Installing proxy npm dependencies...')
        subprocess.run(['npm', 'install', '--omit=dev', '--silent'], cwd=PROXY_DIR, check=True)
    os.makedirs(os.path.dirname(PROXY_ZIP), exist_ok=True)
    if os.path.exists(PROXY_ZIP):
        os.remove(PROXY_ZIP)
    subprocess.run(['zip', '-rq', PROXY_ZIP, 'index.mjs', 'package.json', 'node_modules'], cwd=PROXY_DIR, check=True)
    with open(PROXY_ZIP, 'rb') as f:
        digest = hashlib.sha256(f.read()).hexdigest()[:16]
    s3_key = f'proxy-{digest}.zip'
    return s3_key, digest

# 1. Bucket for the Lambda zip
ensure_assets_bucket()

# 2. Build + upload proxy.zip (key is content-addressed so identical builds are cache-friendly)
s3_key, digest = build_proxy_zip()
subprocess.run(['aws', 's3', 'cp', PROXY_ZIP, f's3://{ASSETS_BUCKET}/{s3_key}', '--region', AWS_REGION], check=True)
print(f'Proxy zip uploaded: s3://{ASSETS_BUCKET}/{s3_key} (sha256:{digest})')

# 3. Create or update the CFN stack with the new asset reference
params = [
    f'ParameterKey=ProxyAssetBucketParam,ParameterValue={ASSETS_BUCKET}',
    f'ParameterKey=ProxyAssetKeyParam,ParameterValue={s3_key}',
]
current = stack_status()
if current is None:
    print(f'Creating stack {STACK_NAME} (CloudFront takes ~5 min)...')
    subprocess.run(
        ['aws', 'cloudformation', 'create-stack', '--stack-name', STACK_NAME,
         '--template-body', f'file://{CFN_TEMPLATE}',
         '--capabilities', 'CAPABILITY_NAMED_IAM', '--region', AWS_REGION,
         '--parameters', *params],
        check=True
    )
    subprocess.run(['aws', 'cloudformation', 'wait', 'stack-create-complete',
                    '--stack-name', STACK_NAME, '--region', AWS_REGION], check=True)
    print(f'Stack created: {stack_status()}')
else:
    print(f'Stack {STACK_NAME} already exists: {current}. Updating with new proxy zip...')
    upd = subprocess.run(
        ['aws', 'cloudformation', 'update-stack', '--stack-name', STACK_NAME,
         '--template-body', f'file://{CFN_TEMPLATE}',
         '--capabilities', 'CAPABILITY_NAMED_IAM', '--region', AWS_REGION,
         '--parameters', *params],
        capture_output=True, text=True
    )
    if upd.returncode == 0:
        subprocess.run(['aws', 'cloudformation', 'wait', 'stack-update-complete',
                        '--stack-name', STACK_NAME, '--region', AWS_REGION], check=True)
        print(f'Stack updated: {stack_status()}')
    elif 'No updates are to be performed' in (upd.stderr or ''):
        print('No updates needed.')
    else:
        raise RuntimeError(f'update-stack failed: {upd.stderr or upd.stdout}')

upload: ../deploy/.build/proxy.zip to s3://agui-bedrock-agentcore-assets-us-east-1-<ACCOUNT-ID>/proxy-da45974dee0ff0c5.zip
Proxy zip uploaded: s3://agui-bedrock-agentcore-assets-us-east-1-<ACCOUNT-ID>/proxy-da45974dee0ff0c5.zip (sha256:da45974dee0ff0c5)
Stack agui-bedrock-agentcore already exists: UPDATE_COMPLETE. Updating with new proxy zip...
Stack updated: UPDATE_COMPLETE


In [33]:
# Direct test of the Lambda Function URL — bypasses CloudFront so we can
# isolate whether streaming works at the Lambda layer before involving CDN
# cache/buffering behavior.
#
# Each printed line shows the wall-clock offset since the request started.
# If streaming works: TEXT_MESSAGE_CONTENT events should arrive spaced out
# (typically 30-200 ms apart). If buffering is happening somewhere, they'll
# all show the same timestamp at the end of the run.
import boto3, json, time, urllib3, re, subprocess
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

TEST_PROMPT = 'Look up customer CUST-001 and CUST-003, compare their annual spend'

# 1. Make sure the Lambda has AGENT_RUNTIME_ARN set. On a fresh deploy this
#    isn't populated yet (wire-frontend does it later), but the test needs it,
#    so populate it here directly from `agentcore status --json`.
status_proc = subprocess.run(
    ['agentcore', 'status', '--target', 'default', '--type', 'agent', '--json'],
    capture_output=True, text=True
)
if status_proc.returncode != 0:
    raise RuntimeError(f'agentcore status failed: {status_proc.stderr or status_proc.stdout}')
m = re.search(r'\{.*\}', status_proc.stdout, re.DOTALL)
status = json.loads(m.group(0))
agent_entry = next((r for r in status.get('resources', []) if r.get('resourceType') == 'agent'), None)
if not agent_entry or agent_entry.get('deploymentState') != 'deployed':
    raise RuntimeError(f'Agent not deployed (state={agent_entry and agent_entry.get("deploymentState")}). Run the launch cell first.')
agent_arn = agent_entry.get('identifier') or agent_entry.get('arn') or agent_entry.get('agentRuntimeArn')

lam = boto3.client('lambda', region_name=AWS_REGION)
current_env = lam.get_function_configuration(FunctionName=LAMBDA_FUNCTION_NAME)['Environment']['Variables']
if current_env.get('AGENT_RUNTIME_ARN') != agent_arn:
    print(f'Setting AGENT_RUNTIME_ARN on {LAMBDA_FUNCTION_NAME}...')
    lam.update_function_configuration(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Environment={'Variables': {**current_env, 'AGENT_RUNTIME_ARN': agent_arn, 'REGION': AWS_REGION}}
    )
    # Wait for the update to propagate before invoking
    waiter = lam.get_waiter('function_updated_v2')
    waiter.wait(FunctionName=LAMBDA_FUNCTION_NAME)
    print(f'  Updated. ARN: {agent_arn}')

# 2. Read FunctionUrl from stack outputs
cfn = boto3.client('cloudformation', region_name=AWS_REGION)
stack = cfn.describe_stacks(StackName=STACK_NAME)['Stacks'][0]
outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}
function_url = outputs.get('ProxyFunctionUrl')
if not function_url:
    raise RuntimeError('ProxyFunctionUrl not found in stack outputs. Re-run deploy-infra.')
url = function_url.rstrip('/') + '/invocations'
print(f'\nCalling: {url}')

# 3. SigV4-sign the request (Function URL is configured AuthType=AWS_IAM)
creds = boto3.Session().get_credentials().get_frozen_credentials()
body = json.dumps({'prompt': TEST_PROMPT})
req = AWSRequest(method='POST', url=url, data=body,
                 headers={'Content-Type': 'application/json'})
SigV4Auth(creds, 'lambda', AWS_REGION).add_auth(req)

# 4. Stream the response
http = urllib3.PoolManager()
resp = http.request('POST', url, body=body, headers=dict(req.headers),
                    preload_content=False)
print(f'\nHTTP {resp.status}  Content-Type: {resp.headers.get("content-type", "?")}\n')
if resp.status != 200:
    print(resp.read().decode())
else:
    start = time.time()
    buf = b''
    event_count = 0
    last_event_time = start
    gaps = []
    for chunk in resp.stream(amt=1024):
        buf += chunk
        # Parse SSE: events are separated by \n\n; each line starts with 'data: '
        while b'\n\n' in buf:
            raw, buf = buf.split(b'\n\n', 1)
            for line in raw.splitlines():
                if not line.startswith(b'data: '):
                    continue
                now = time.time()
                gap = now - last_event_time
                gaps.append(gap)
                last_event_time = now
                try:
                    ev = json.loads(line[6:])
                    et = ev.get('type', '?')
                    extra = {k: v for k, v in ev.items() if k not in ('type', 'eventId', 'timestamp')}
                    extra_str = json.dumps(extra)[:80] if extra else ''
                    print(f'[+{now-start:6.2f}s | gap {gap*1000:5.0f}ms] {et:25} {extra_str}')
                except json.JSONDecodeError:
                    print(f'[+{now-start:6.2f}s] (non-JSON) {line[:80]!r}')
                event_count += 1
    resp.release_conn()

    # Verdict
    print()
    print(f'Total: {event_count} events in {time.time()-start:.2f}s')
    if gaps:
        # Skip the first gap (it's just startup latency, not a streaming signal)
        mid_gaps = gaps[1:] if len(gaps) > 1 else gaps
        max_gap = max(mid_gaps) * 1000
        # If every event arrives within 5ms of the last, it's almost certainly buffered
        burst_count = sum(1 for g in mid_gaps if g < 0.005)
        burst_ratio = burst_count / len(mid_gaps) if mid_gaps else 0
        print(f'Max inter-event gap: {max_gap:.0f}ms')
        print(f'Events within 5ms of previous: {burst_count}/{len(mid_gaps)} ({burst_ratio:.0%})')
        if burst_ratio > 0.8:
            print('\n⚠️  Looks like the response is buffered — events are arriving in a burst.')
            print('   Check that ProxyLambdaUrl has InvokeMode=RESPONSE_STREAM and the')
            print('   handler is wrapped in awslambda.streamifyResponse.')
        else:
            print('\n✅ Streaming works — events arrive incrementally.')


Calling: https://sw6p6ptgzs52ehjsmlr3nghtlu0iempj.lambda-url.us-east-1.on.aws/invocations

HTTP 200  Content-Type: text/event-stream

[+  0.00s | gap     0ms] RUN_STARTED               {"runId": "6c51d2a9-49a5-4557-87c6-73e3fd034f9e"}
[+  0.00s | gap     0ms] STATE_SNAPSHOT            {"snapshot": {"status": "processing", "tools_invoked": []}}
[+  0.76s | gap   757ms] TEXT_MESSAGE_START        {"messageId": "a1cb865a-591a-41f2-8661-6a11da10bc01", "role": "assistant"}
[+  0.76s | gap     0ms] TEXT_MESSAGE_CONTENT      {"messageId": "a1cb865a-591a-41f2-8661-6a11da10bc01", "delta": "Sure"}
[+  0.86s | gap   103ms] TEXT_MESSAGE_CONTENT      {"messageId": "a1cb865a-591a-41f2-8661-6a11da10bc01", "delta": "!"}
[+  0.88s | gap    20ms] TEXT_MESSAGE_CONTENT      {"messageId": "a1cb865a-591a-41f2-8661-6a11da10bc01", "delta": " Let"}
[+  0.90s | gap    17ms] TEXT_MESSAGE_CONTENT      {"messageId": "a1cb865a-591a-41f2-8661-6a11da10bc01", "delta": " me look up both
[+  0.92s | gap    26ms] TEXT_ME

In [34]:
# Same streaming test, but through CloudFront. Run AFTER test-lambda-streaming
# so you can compare gaps side-by-side. If Lambda-direct shows 30-200ms gaps
# and CloudFront shows zero/near-zero gaps in a burst at the end, CloudFront
# is buffering the response and we'll need to fix the cache behavior.
import boto3, json, time, urllib3

TEST_PROMPT = 'Look up customer CUST-001 and CUST-003, compare their annual spend'

# Read the CloudFront URL from stack outputs
cfn = boto3.client('cloudformation', region_name=AWS_REGION)
stack = cfn.describe_stacks(StackName=STACK_NAME)['Stacks'][0]
outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}
frontend_url = outputs.get('FrontendUrl')
if not frontend_url:
    raise RuntimeError('FrontendUrl not found in stack outputs. Re-run deploy-infra.')
url = frontend_url.rstrip('/') + '/invocations'
print(f'Calling: {url}')
print('(no SigV4 — CloudFront OAC signs the origin request, viewer is anonymous)')
print('(IMPORTANT: your IP must match the AllowedIPv4CIDR parameter)')

# CloudFront accepts the request unsigned; OAC signs the origin request to Lambda
body = json.dumps({'prompt': TEST_PROMPT})
http = urllib3.PoolManager()
resp = http.request('POST', url, body=body,
                    headers={'Content-Type': 'application/json'},
                    preload_content=False)
print(f'\nHTTP {resp.status}  Content-Type: {resp.headers.get("content-type", "?")}\n')
# Show all response headers — useful to spot Content-Encoding (gzip), X-Cache, etc.
for k, v in resp.headers.items():
    print(f'  {k}: {v}')
print()

if resp.status != 200:
    print(resp.read().decode()[:500])
else:
    start = time.time()
    buf = b''
    event_count = 0
    last_event_time = start
    gaps = []
    for chunk in resp.stream(amt=1024):
        buf += chunk
        while b'\n\n' in buf:
            raw, buf = buf.split(b'\n\n', 1)
            for line in raw.splitlines():
                if not line.startswith(b'data: '):
                    continue
                now = time.time()
                gap = now - last_event_time
                gaps.append(gap)
                last_event_time = now
                try:
                    ev = json.loads(line[6:])
                    et = ev.get('type', '?')
                    extra = {k: v for k, v in ev.items() if k not in ('type', 'eventId', 'timestamp')}
                    extra_str = json.dumps(extra)[:80] if extra else ''
                    print(f'[+{now-start:6.2f}s | gap {gap*1000:5.0f}ms] {et:25} {extra_str}')
                except json.JSONDecodeError:
                    print(f'[+{now-start:6.2f}s] (non-JSON) {line[:80]!r}')
                event_count += 1
    resp.release_conn()

    print()
    print(f'Total: {event_count} events in {time.time()-start:.2f}s')
    if gaps:
        mid_gaps = gaps[1:] if len(gaps) > 1 else gaps
        max_gap = max(mid_gaps) * 1000
        # Stricter threshold for CloudFront: if more than half the events arrive
        # within 5ms of each other, it's likely buffered
        burst_count = sum(1 for g in mid_gaps if g < 0.005)
        burst_ratio = burst_count / len(mid_gaps) if mid_gaps else 0
        # Also check time-to-first-event — buffering shows up as a long initial delay
        time_to_first = gaps[0] * 1000 if gaps else 0
        print(f'Time to first event: {time_to_first:.0f}ms')
        print(f'Max inter-event gap: {max_gap:.0f}ms')
        print(f'Events within 5ms of previous: {burst_count}/{len(mid_gaps)} ({burst_ratio:.0%})')
        if burst_ratio > 0.7:
            print('\n⚠️  CloudFront appears to be buffering — most events arrive in a burst.')
            print('   Likely fixes:')
            print('   1. Set Compress: false on the /invocations cache behavior (already done)')
            print('   2. Use a custom OriginRequestPolicy that does NOT forward Accept-Encoding')
            print('      (so the Lambda runtime does not gzip its response)')
        elif time_to_first > 3000 and event_count < 20:
            print(f'\n⚠️  First event took {time_to_first:.0f}ms — likely waiting for full response.')
        else:
            print('\n✅ CloudFront is streaming the response incrementally.')

Calling: https://d3do5hm7qme3b.cloudfront.net/invocations
(no SigV4 — CloudFront OAC signs the origin request, viewer is anonymous)
(IMPORTANT: your IP must match the AllowedIPv4CIDR parameter)

HTTP 403  Content-Type: application/json

  Content-Type: application/json
  Content-Length: 192
  Connection: keep-alive
  Date: Sat, 16 May 2026 05:02:59 GMT
  x-amzn-ErrorType: InvalidSignatureException
  x-amzn-RequestId: 863d0fe8-1322-4fba-946d-612c25f6a83e
  X-Cache: Error from cloudfront
  Via: 1.1 31f49a22e12e4f9959a8e370a5ed48de.cloudfront.net (CloudFront)
  X-Amz-Cf-Pop: LAX53-P6
  Alt-Svc: h3=":443"; ma=86400
  X-Amz-Cf-Id: mgD7atZXKEZay95um2ruYs7YxHiL3S9uZfkoWWplHr8IeokzzcKFKA==

{"message":"The request signature we calculated does not match the signature you provided. Check your AWS Secret Access Key and signing method. Consult the service documentation for details."}


In [31]:
import boto3, subprocess, json, re, time

# 1. Read CloudFormation stack outputs
cfn = boto3.client('cloudformation', region_name=AWS_REGION)
try:
    stack = cfn.describe_stacks(StackName=STACK_NAME)['Stacks'][0]
except cfn.exceptions.ClientError as e:
    raise RuntimeError(f'Stack {STACK_NAME} not found. Run the deploy-infra cell first.') from e
outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}
bucket = outputs.get('FrontendBucketName')
dist_id = outputs.get('CloudFrontDistributionId')
frontend_url = outputs.get('FrontendUrl', 'N/A')

# 2. Read agent ARN via `agentcore status --json`
status_proc = subprocess.run(
    ['agentcore', 'status', '--target', 'default', '--type', 'agent', '--json'],
    capture_output=True, text=True
)
agent_arn = None
if status_proc.returncode == 0:
    m = re.search(r'\{.*\}', status_proc.stdout, re.DOTALL)
    if m:
        status = json.loads(m.group(0))
        agent_entry = next((r for r in status.get('resources', []) if r.get('resourceType') == 'agent'), None)
        if agent_entry and agent_entry.get('deploymentState') == 'deployed':
            agent_arn = agent_entry.get('identifier') or agent_entry.get('arn') or agent_entry.get('agentRuntimeArn')
if not agent_arn:
    raise RuntimeError('Could not determine agent ARN. Run `agentcore status --target default` to check deployment.')

# 3. Update Lambda proxy with the agent ARN
lambda_client = boto3.client('lambda', region_name=AWS_REGION)
lambda_client.update_function_configuration(
    FunctionName=LAMBDA_FUNCTION_NAME,
    Environment={'Variables': {'AGENT_RUNTIME_ARN': agent_arn, 'REGION': AWS_REGION}}
)
print(f'Lambda proxy updated: {agent_arn}')

# 4. Upload frontend index.html to S3
if not os.path.exists(FRONTEND_HTML):
    raise FileNotFoundError(f'Frontend not found at {FRONTEND_HTML}')
s3 = boto3.client('s3', region_name=AWS_REGION)
s3.upload_file(FRONTEND_HTML, bucket, 'index.html', ExtraArgs={'ContentType': 'text/html'})
print(f'Frontend uploaded:    s3://{bucket}/index.html')

# 5. Invalidate CloudFront cache so the new HTML is served immediately
cf = boto3.client('cloudfront')
inv = cf.create_invalidation(
    DistributionId=dist_id,
    InvalidationBatch={
        'Paths': {'Quantity': 1, 'Items': ['/*']},
        'CallerReference': f'wire-frontend-{int(time.time())}'
    }
)
print(f'CloudFront invalidation submitted: {inv["Invalidation"]["Id"]}')

# 6. Print the URL
print(f'\n{"="*50}')
print(f'AG-UI FRONTEND: {frontend_url}')
print(f'{"="*50}')
print('\nTry these queries:')
print('  1. Check warehouse-east for low stock items, calculate restock cost, send a slack alert')
print('  2. Look up customer CUST-001 and CUST-003, compare their spend')
print('  3. Analyze error trends for 7 days, analyze sales for 30 days, generate a report')

Lambda proxy updated: arn:aws:bedrock-agentcore:us-east-1:<ACCOUNT-ID>:runtime/aguisampleagent_aguisampleagent-esfpXICGVN
Frontend uploaded:    s3://agui-bedrock-agentcore-ui-us-east-1-<ACCOUNT-ID>/index.html
CloudFront invalidation submitted: I8P730CHT4FVUSVSOH0633MTW4

AG-UI FRONTEND: https://d3do5hm7qme3b.cloudfront.net

Try these queries:
  1. Check warehouse-east for low stock items, calculate restock cost, send a slack alert
  2. Look up customer CUST-001 and CUST-003, compare their spend
  3. Analyze error trends for 7 days, analyze sales for 30 days, generate a report


## Cleanup

In [ ]:
# Uncomment to clean up

# # Delete AgentCore deployment
# !agentcore destroy --target default -y

# # Delete supporting infrastructure
# os.chdir(NOTEBOOK_DIR)
# !aws s3 rm s3://$(aws cloudformation describe-stacks --stack-name {STACK_NAME} --region {AWS_REGION} --query 'Stacks[0].Outputs[?OutputKey==`FrontendBucketName`].OutputValue' --output text)/ --recursive
# !aws cloudformation delete-stack --stack-name {STACK_NAME} --region {AWS_REGION}

# # Clean up local files
# !rm -rf {AGENT_NAME}/
# print('Done')